# GraphAgg ET stack


This notebook produces the reported **GraphAgg ET stack** on the frozen split directly from:
- `elliptic_txs_features.csv`
- `elliptic_txs_classes.csv`
- `elliptic_txs_edgelist.csv`

It uses the same stage-1 ET-600 anchor and stage-2 graph-aggregate ET configuration used in the final project artifacts.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('..')
RAW_DIR = PROJECT_DIR / 'Dataset' / 'raw' / 'elliptic_bitcoin_dataset'

OUT_DIR = PROJECT_DIR / 'outputs' / 'graphagg_et_stack'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_CSV = RAW_DIR / 'elliptic_txs_features.csv'
CLASSES_CSV = RAW_DIR / 'elliptic_txs_classes.csv'
EDGELIST_CSV = RAW_DIR / 'elliptic_txs_edgelist.csv'

OUT_JSON = OUT_DIR / 'newsplit_32_37_42_graphagg_stack_rebuilt_summary.json'
OUT_CSV = OUT_DIR / 'newsplit_32_37_42_graphagg_stack_rebuilt_stepwise.csv'

In [2]:
import json, time
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

start = time.time()

def log(*args):
    pass

# Load features
feat = pd.read_csv(FEATURES_CSV, header=None)
feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(165)]

classes = pd.read_csv(CLASSES_CSV)
df = feat.merge(classes, on='txId', how='left')
df['y'] = pd.to_numeric(df['class'], errors='coerce').map({1: 1, 2: 0})

n = len(df)
log('loaded rows', n, 'elapsed', round(time.time() - start, 2), 'sec')

# Map edges to row indices
idx_map = pd.Series(np.arange(n, dtype=np.int32), index=df['txId'].values)
edges = pd.read_csv(EDGELIST_CSV)
src = idx_map.loc[edges['txId1']].to_numpy(np.int32)
dst = idx_map.loc[edges['txId2']].to_numpy(np.int32)
log('mapped edges', len(src), 'elapsed', round(time.time() - start, 2), 'sec')

X_all = df[[f'f{i}' for i in range(165)]].to_numpy(np.float32)
X_local = df[[f'f{i}' for i in range(93)]].to_numpy(np.float32)
y = df['y'].to_numpy()
t = df['time_step'].to_numpy(np.int16)
labeled = ~np.isnan(y)

# Stage-1 anchor: rolling 8-step ET-600 OOF for train t=9...32
p_anchor = np.full(n, np.nan, dtype=np.float32)
fit_stats = []

for cur_t in range(9, 33):
    fit_mask = labeled & (t >= cur_t - 8) & (t <= cur_t - 1)
    pred_mask = (t == cur_t)

    model = ExtraTreesClassifier(
        n_estimators=600,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=7,
        n_jobs=-1,
    )

    t0 = time.time()
    model.fit(X_all[fit_mask], y[fit_mask].astype(int))
    fit_sec = time.time() - t0

    p_anchor[pred_mask] = model.predict_proba(X_all[pred_mask])[:, 1].astype(np.float32)

    fit_stats.append({
        't': cur_t,
        'fit_samples': int(fit_mask.sum()),
        'pred_samples': int(pred_mask.sum()),
        'fit_sec': fit_sec
    })

    log(
        'stage1 t', cur_t,
        'fit_samples', int(fit_mask.sum()),
        'pred_samples', int(pred_mask.sum()),
        'fit_sec', round(fit_sec, 2),
        'elapsed', round(time.time() - start, 2)
    )

# Full-train ET-600 for val/test t=33...42
full_train_mask = labeled & (t >= 1) & (t <= 32)

model_full = ExtraTreesClassifier(
    n_estimators=600,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=7,
    n_jobs=-1,
)

t0 = time.time()
model_full.fit(X_all[full_train_mask], y[full_train_mask].astype(int))
full_fit_sec = time.time() - t0

pred_mask = (t >= 33) & (t <= 42)
p_anchor[pred_mask] = model_full.predict_proba(X_all[pred_mask])[:, 1].astype(np.float32)

log('stage1 full-train fit_sec', round(full_fit_sec, 2), 'elapsed', round(time.time() - start, 2))

need_mask = (t >= 13) & (t <= 42)
missing = int(np.isnan(p_anchor[need_mask]).sum())
log('anchor missing on t13..42', missing)
if missing:
    raise RuntimeError(f'Missing anchor predictions on required rows: {missing}')

# Self score features
EPS = 1e-6
p_anchor_safe = np.where(np.isnan(p_anchor), 0.5, p_anchor).astype(np.float32)
p_clip = np.clip(p_anchor_safe, EPS, 1 - EPS)
logit = np.log(p_clip / (1 - p_clip)).astype(np.float32)
abslogit = np.abs(logit).astype(np.float32)

# Graph aggregate features from anchor probabilities
in_deg = np.bincount(dst, minlength=n).astype(np.float32)
out_deg = np.bincount(src, minlength=n).astype(np.float32)
tot_deg = in_deg + out_deg

out_sum = np.bincount(src, weights=p_clip[dst], minlength=n).astype(np.float32)
out_cnt05 = np.bincount(src, weights=(p_clip[dst] > 0.5).astype(np.float32), minlength=n).astype(np.float32)
out_cnt09 = np.bincount(src, weights=(p_clip[dst] > 0.9).astype(np.float32), minlength=n).astype(np.float32)
out_max = np.zeros(n, dtype=np.float32)
np.maximum.at(out_max, src, p_clip[dst])

in_sum = np.bincount(dst, weights=p_clip[src], minlength=n).astype(np.float32)
in_cnt05 = np.bincount(dst, weights=(p_clip[src] > 0.5).astype(np.float32), minlength=n).astype(np.float32)
in_cnt09 = np.bincount(dst, weights=(p_clip[src] > 0.9).astype(np.float32), minlength=n).astype(np.float32)
in_max = np.zeros(n, dtype=np.float32)
np.maximum.at(in_max, dst, p_clip[src])

in_mean = np.divide(in_sum, in_deg, out=np.zeros_like(in_sum), where=in_deg > 0)
out_mean = np.divide(out_sum, out_deg, out=np.zeros_like(out_sum), where=out_deg > 0)
all_mean = np.divide(in_sum + out_sum, tot_deg, out=np.zeros_like(in_sum), where=tot_deg > 0)
all_max = np.maximum(in_max, out_max)
all_cnt05 = in_cnt05 + out_cnt05
all_cnt09 = in_cnt09 + out_cnt09

graph_feats = np.column_stack([
    np.log1p(in_deg), np.log1p(out_deg), np.log1p(tot_deg),
    in_mean, in_max, in_cnt05, in_cnt09,
    out_mean, out_max, out_cnt05, out_cnt09,
    all_mean, all_max, all_cnt05, all_cnt09,
]).astype(np.float32)

log('graph features built', graph_feats.shape, 'elapsed', round(time.time() - start, 2))

# Validation-selected stage-2 model:
# local93 + self score + graph aggregates; train timesteps 13..32; ET-300; min_leaf=3
X_stage2 = np.concatenate([
    X_local,
    p_clip[:, None],
    logit[:, None],
    abslogit[:, None],
    graph_feats,
], axis=1).astype(np.float32)

train2_mask = labeled & (t >= 13) & (t <= 32)
val_mask = labeled & (t >= 33) & (t <= 37)
test_mask = labeled & (t >= 38) & (t <= 42)

model_stage2 = ExtraTreesClassifier(
    n_estimators=300,
    max_features='sqrt',
    min_samples_leaf=3,
    class_weight='balanced_subsample',
    random_state=7,
    n_jobs=-1,
)

t0 = time.time()
model_stage2.fit(X_stage2[train2_mask], y[train2_mask].astype(int))
stage2_fit_sec = time.time() - t0
log('stage2 fit_sec', round(stage2_fit_sec, 2), 'elapsed', round(time.time() - start, 2))

val_p = model_stage2.predict_proba(X_stage2[val_mask])[:, 1]
test_p = model_stage2.predict_proba(X_stage2[test_mask])[:, 1]

val_pr = float(average_precision_score(y[val_mask], val_p))
test_pr = float(average_precision_score(y[test_mask], test_p))
test_roc = float(roc_auc_score(y[test_mask], test_p))
test_brier = float(brier_score_loss(y[test_mask], test_p))

# Stepwise table
rows = []
for step in range(38, 43):
    m = labeled & (t == step)
    yp = y[m]
    pp = model_stage2.predict_proba(X_stage2[m])[:, 1]
    rows.append({
        'step': step,
        'labeled': int(m.sum()),
        'illicit': int(yp.sum()),
        'illicit_rate': float(yp.mean()),
        'graphagg_pr_auc': float(average_precision_score(yp, pp)),
        'graphagg_roc_auc': float(roc_auc_score(yp, pp)),
        'graphagg_brier': float(brier_score_loss(yp, pp)),
    })

step_df = pd.DataFrame(rows)
step_df.to_csv(OUT_CSV, index=False)

summary = {
    'active_split': {'train': [1, 32], 'val': [33, 37], 'test': [38, 42]},
    'stage1_anchor': {
        'model': 'ExtraTreesClassifier',
        'n_estimators': 600,
        'max_features': 'sqrt',
        'class_weight': 'balanced_subsample',
        'random_state': 7,
        'oof_scheme': 'rolling_8_step_time_forward',
        'oof_train_steps': [9, 32],
        'full_train_predict_steps': [33, 42],
        'fit_stats_head': fit_stats[:5],
        'fit_stats_tail': fit_stats[-5:],
        'full_train_fit_sec': full_fit_sec,
    },
    'stage2_model': {
        'mode': 'local93_score_graph',
        'train_steps': [13, 32],
        'n_estimators': 300,
        'min_samples_leaf': 3,
        'max_features': 'sqrt',
        'class_weight': 'balanced_subsample',
        'random_state': 7,
        'n_features': int(X_stage2.shape[1]),
    },
    'metrics': {
        'val_pr_auc': val_pr,
        'test_pr_auc': test_pr,
        'test_roc_auc': test_roc,
        'test_brier': test_brier,
    },
    'elapsed_sec_total': time.time() - start,
}

with open(OUT_JSON, 'w') as f:
    json.dump(summary, f, indent=2)

print(
    f"METRICS val_pr {val_pr:.6f} "
    f"test_pr {test_pr:.6f} "
    f"test_roc {test_roc:.6f} "
    f"test_brier {test_brier:.5f}"
)

METRICS val_pr 0.953235 test_pr 0.904974 test_roc 0.964947 test_brier 0.02489
